# VibeShift — RTX 4090 Training
Flow-matching genre transformation · **tqdm** progress bars · **W&B** experiment tracking

| Step | Cell |
|---|---|
| 1 | Install deps |
| 2 | Imports + GPU check |
| 3 | Config (edit paths here) |
| 4 | W&B login |
| 5 | Data loaders |
| 6 | Build model |
| 7 | Optimizer + scheduler |
| 8 | **Train** |
| 9 | Plot results |
| 10 | Gradient norms |
| 11 | Quick inference test |

In [ ]:
# Cell 1 — Install deps (run once, then restart kernel)
%pip install -q wandb tqdm

In [ ]:
# Cell 2 — Imports + GPU check
import sys, os

# Resolve workspace root (works from notebooks/training/)
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
import numpy as np
import matplotlib.pyplot as plt
import wandb
from tqdm.auto import tqdm

from models.dit import DiT
from models.flow import FlowMatching
from training.training import Trainer, TrainingConfig
from training.dataloader import create_dataloader

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU     : {props.name}")
    print(f"VRAM    : {props.total_memory / 1e9:.1f} GB")
    print(f"BF16    : {torch.cuda.is_bf16_supported()}")

## ⚙️ Config
Edit **PATHS** and **MODEL_CONFIG** here. Hyperparameter defaults are already tuned for RTX 4090 (24 GB).

In [ ]:

# Cell 3 — Config

# ─── Paths ────────────────────────────────────────────────────────────────────
SOURCE_DIR     = os.path.join(ROOT, "data", "output", "synth_dac")
TARGET_DIR     = os.path.join(ROOT, "data", "output", "classical_dac")
CHECKPOINT_DIR = os.path.join(ROOT, "checkpoints")
LOG_DIR        = os.path.join(ROOT, "logs")
RUN_NAME       = "synth-to-punk-4090"

# ─── Split ratios ─────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10   # must sum to 1.0

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-6, "Ratios must sum to 1.0"

os.makedirs(LOG_DIR, exist_ok=True)

# ─── Model architecture ───────────────────────────────────────────────────────
MODEL_CONFIG = dict(
    input_dim  = 768,   # DAC latent dim — overridden from data below
    embed_dim  = 512,
    num_blocks = 8,
    num_heads  = 8,
    num_genres = 3,
    hidden_dim = 2048,
    dropout    = 0.1,
)

# ─── Training hyperparameters ─────────────────────────────────────────────────
config = TrainingConfig(
    num_epochs             = 200,
    batch_size             = 16,      # 4090 handles 16–32 comfortably with AMP
    learning_rate          = 1e-3,
    weight_decay           = 0.01,    # AdamW standard
    gradient_clip          = 1.0,
    accumulation_steps     = 1,       # set 2 for effective batch 32
    early_stopping_patience= 20,
    monitor_gradients      = True,
    use_amp                = True,    # FP16/BF16 ~2x throughput on 4090
    device                 = "cuda" if torch.cuda.is_available() else "cpu",
    wandb_project          = "vibeshift",
    wandb_run_name         = RUN_NAME,
    num_workers            = 8,
    pin_memory             = True,
)

print(config)
print(f"\nSplit  : train={TRAIN_RATIO:.0%}  val={VAL_RATIO:.0%}  test={TEST_RATIO:.0%}  (sequential)")


## 📊 W&B Login

In [ ]:
# Cell 4 — W&B login
# Opens a browser prompt or uses WANDB_API_KEY env var silently
wandb.login()

## 📂 Data

In [ ]:

# Cell 5a — 80 / 10 / 10 dataset split (sequential)
from torch.utils.data import Subset

full_dataset = create_dataloader(
    source_dir  = SOURCE_DIR,
    target_dir  = TARGET_DIR,
    batch_size  = 1,          # we only need the dataset object here
    num_workers = 0,
)[1]   # [1] → the LatentPairDataset

n_total = len(full_dataset)
n_train = int(n_total * TRAIN_RATIO)
n_val   = int(n_total * VAL_RATIO)
n_test  = n_total - n_train - n_val   # absorbs rounding remainder

train_set = Subset(full_dataset, range(0, n_train))
val_set   = Subset(full_dataset, range(n_train, n_train + n_val))
test_set  = Subset(full_dataset, range(n_train + n_val, n_total))

print(f"Total samples : {n_total}")
print(f"Train  (80 %) : {len(train_set)}")
print(f"Val    (10 %) : {len(val_set)}")
print(f"Test   (10 %) : {len(test_set)}")


In [ ]:

# Cell 5b — Build DataLoaders from the split subsets
from torch.utils.data import DataLoader
from training.dataloader import default_collate_with_dynamic_padding

def make_loader(subset, shuffle: bool, drop_last: bool) -> DataLoader:
    return DataLoader(
        subset,
        batch_size  = config.batch_size,
        shuffle     = shuffle,
        num_workers = config.num_workers,
        pin_memory  = config.pin_memory,
        drop_last   = drop_last,
        collate_fn  = default_collate_with_dynamic_padding,
    )

train_loader = make_loader(train_set, shuffle=True,  drop_last=True)
val_loader   = make_loader(val_set,   shuffle=False, drop_last=False)
test_loader  = make_loader(test_set,  shuffle=False, drop_last=False)

info = full_dataset.get_info()
print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")
print(f"Embedding dim : {info['embedding_dim']}")

# Sync actual embedding dim to model config
MODEL_CONFIG["input_dim"] = info["embedding_dim"]


## 🏗️ Model

In [ ]:
# Cell 6 — Build model
dit   = DiT(**MODEL_CONFIG)
model = FlowMatching(dit)

# Optional: torch.compile gives ~15–30% speedup on 4090 (PyTorch 2+)
# model = torch.compile(model)

n_total    = sum(p.numel() for p in model.parameters())
n_trainable= sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters : {n_total/1e6:.2f}M total  {n_trainable/1e6:.2f}M trainable")
print(f"Model cfg  : {MODEL_CONFIG}")

# Quick VRAM estimate (rough: 4 bytes * params * 4 for optimizer states)
vram_est_gb = (n_trainable * 4 * 4) / 1e9
print(f"Rough VRAM est (model+optim): {vram_est_gb:.1f} GB")

## 🔧 Optimizer & Scheduler
Using **AdamW** with β₂=0.95 (good for transformers) + **10-epoch linear warmup** → **cosine decay**.

In [ ]:
# Cell 7 — Optimizer + scheduler
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = config.learning_rate,
    weight_decay = config.weight_decay,
    betas        = (0.9, 0.95),
)

WARMUP_EPOCHS = 10
COSINE_EPOCHS = config.num_epochs - WARMUP_EPOCHS

warmup_sched = LinearLR(
    optimizer,
    start_factor = 1 / 25,  # initial_lr = peak_lr / 25
    end_factor   = 1.0,
    total_iters  = WARMUP_EPOCHS,
)
cosine_sched = CosineAnnealingLR(
    optimizer,
    T_max   = COSINE_EPOCHS,
    eta_min = 1e-6,
)
scheduler = SequentialLR(
    optimizer,
    schedulers = [warmup_sched, cosine_sched],
    milestones = [WARMUP_EPOCHS],
)

# Preview LR curve
_lrs = []
_opt_preview = torch.optim.AdamW([torch.zeros(1)], lr=config.learning_rate)
_w = LinearLR(_opt_preview, 1/25, 1.0, WARMUP_EPOCHS)
_c = CosineAnnealingLR(_opt_preview, COSINE_EPOCHS, 1e-6)
_s = SequentialLR(_opt_preview, [_w, _c], [WARMUP_EPOCHS])
for _ in range(config.num_epochs):
    _lrs.append(_opt_preview.param_groups[0]["lr"])
    _s.step()

plt.figure(figsize=(10, 3))
plt.plot(_lrs, linewidth=2, color='darkorange')
plt.axvline(WARMUP_EPOCHS, linestyle='--', color='gray', alpha=0.7, label=f'Warmup end (ep {WARMUP_EPOCHS})')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('LR Schedule Preview')
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Peak LR  : {config.learning_rate:.2e}")
print(f"Start LR : {config.learning_rate/25:.2e}")
print(f"Final LR : 1e-6")

## 🚀 Train
Progress bars via **tqdm** · metrics streamed live to **[W&B](https://wandb.ai)**.

In [ ]:
# Cell 8 — Train
trainer = Trainer(
    model          = model,
    optimizer      = optimizer,
    device         = config.device,
    checkpoint_dir = CHECKPOINT_DIR,
    name           = RUN_NAME,
    use_amp        = config.use_amp,
    wandb_project  = config.wandb_project,
    wandb_run_name = config.wandb_run_name,
    wandb_config   = {**config.to_dict(), **MODEL_CONFIG},
)

def loss_fn(x0, x1, genre_ids, mask=None):
    return model.compute_loss(x0, x1, genre_ids, mask)

results = trainer.train(
    train_dataloader        = train_loader,
    num_epochs              = config.num_epochs,
    loss_fn                 = loss_fn,
    val_dataloader          = val_loader,
    scheduler               = scheduler,
    gradient_clip           = config.gradient_clip,
    accumulation_steps      = config.accumulation_steps,
    early_stopping_patience = config.early_stopping_patience,
    log_interval            = config.log_interval,
    monitor_gradients       = config.monitor_gradients,
    save_interval           = 10,
)

## 📈 Loss Curves

In [ ]:
# Cell 9 — Plot training results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"VibeShift — {RUN_NAME}", fontsize=14, fontweight="bold")

# Loss curves
ax = axes[0]
ax.plot(results["epoch_losses"], label="Train", linewidth=2, color="royalblue")
if results["val_losses"]:
    ax.plot(results["val_losses"], label="Val", linewidth=2, color="tomato")
ax.axvline(
    results["best_epoch"], linestyle="--", color="green", alpha=0.7,
    label=f"Best (ep {results['best_epoch']+1})\n{results['best_loss']:.6f}"
)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (MSE)")
ax.set_title("Epoch Loss")
ax.legend(fontsize=8)
ax.set_yscale("log")
ax.grid(True, alpha=0.3)

# LR schedule
ax = axes[1]
ax.plot(results["learning_rates"], linewidth=2, color="darkorange")
ax.set_xlabel("Epoch")
ax.set_ylabel("Learning Rate")
ax.set_title("LR Schedule (actual)")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)

# Batch loss (smoothed)
ax = axes[2]
bl = results["batch_losses"]
if len(bl) >= 50:
    smoothed = np.convolve(bl, np.ones(50) / 50, mode="valid")
    ax.plot(bl, alpha=0.15, color="steelblue", linewidth=0.5)
    ax.plot(range(49, len(bl)), smoothed, color="steelblue", linewidth=2, label="50-step MA")
else:
    ax.plot(bl, color="steelblue", linewidth=1.5)
ax.set_xlabel("Batch")
ax.set_ylabel("Loss")
ax.set_title("Batch Loss")
ax.set_yscale("log")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)

plt.tight_layout()
out_path = os.path.join(LOG_DIR, f"{RUN_NAME}_curves.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out_path}")
print(f"Best loss  : {results['best_loss']:.6f} @ epoch {results['best_epoch']+1}")
print(f"Final loss : {results['epoch_losses'][-1]:.6f}")

## 🔍 Gradient Norms

In [ ]:
# Cell 10 — Gradient norm analysis
norms = results["gradient_norms"]
if norms:
    plt.figure(figsize=(12, 4))
    window = min(50, len(norms) // 5)
    if len(norms) >= window:
        smoothed = np.convolve(norms, np.ones(window) / window, mode="valid")
        plt.plot(norms, alpha=0.2, color="purple", linewidth=0.5)
        plt.plot(range(window - 1, len(norms)), smoothed, color="purple", linewidth=2, label=f"{window}-step MA")
    else:
        plt.plot(norms, color="purple", linewidth=1.5)
    plt.axhline(
        config.gradient_clip, linestyle="--", color="red", alpha=0.7,
        label=f"Clip threshold ({config.gradient_clip})"
    )
    plt.xlabel("Optimizer Step")
    plt.ylabel("Gradient Norm (L2)")
    plt.title("Gradient Norm During Training")
    plt.yscale("log")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f"Mean : {np.mean(norms):.4f}")
    print(f"Max  : {np.max(norms):.4f}")
    print(f"Min  : {np.min(norms):.6f}")
    clipped = sum(1 for n in norms if n > config.gradient_clip)
    print(f"Clipped steps: {clipped}/{len(norms)} ({100*clipped/len(norms):.1f}%)")
else:
    print("Gradient monitoring was disabled (set monitor_gradients=True in config).")

## 🎵 Quick Inference Test
Load the best checkpoint and transform one latent to verify the model is sane.

In [ ]:
# Cell 11 — Inference sanity check
import pathlib

best_ckpt = pathlib.Path(CHECKPOINT_DIR) / RUN_NAME / "best.pt"
print(f"Loading: {best_ckpt}")

if best_ckpt.exists():
    trainer.load_checkpoint(str(best_ckpt), load_optimizer=False)
    model.eval()

    batch = next(iter(val_loader))
    x0, x1, genre_ids, mask = Trainer._unpack_batch(batch)
    x0_single       = x0[:1].to(config.device)
    x1_single       = x1[:1].to(config.device)
    genre_ids_single = genre_ids[:1].to(config.device)

    with torch.no_grad():
        x_euler = model.sample_euler(x0_single, genre_ids_single, num_steps=50)
        x_heun  = model.sample_heun(x0_single,  genre_ids_single, num_steps=25)

    mse_euler = (x_euler - x1_single).pow(2).mean().item()
    mse_heun  = (x_heun  - x1_single).pow(2).mean().item()
    mse_input = (x0_single - x1_single).pow(2).mean().item()  # baseline (no transform)

    print(f"\nShape in  : {x0_single.shape}")
    print(f"Shape out : {x_euler.shape}")
    print(f"\nMSE vs target:")
    print(f"  Input (no transform) : {mse_input:.6f}  ← baseline")
    print(f"  Euler (50 steps)     : {mse_euler:.6f}")
    print(f"  Heun  (25 steps)     : {mse_heun:.6f}")
    if mse_euler < mse_input:
        print("\n✓ Model improves over identity transform")
    else:
        print("\n⚠ Model worse than identity — may need more training")
else:
    print(f"Checkpoint not found at {best_ckpt}\nRun training (Cell 8) first.")